<a href="https://colab.research.google.com/github/k-nikhil-coder/100-day-ai-engineer/blob/main/day15_xgboost_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import xgboost as xgb

# 1. Load data
data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target

# 2. Train-test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {X_train.shape[0]} | Testing samples: {X_test.shape[0]}")

Training samples: 455 | Testing samples: 114


In [2]:
# Baseline model with default parameters
baseline_model = xgb.XGBClassifier(
    eval_metric="logloss",
    random_state=42
)
baseline_model.fit(X_train, y_train)

# Evaluate
baseline_preds = baseline_model.predict(X_test)
baseline_acc = accuracy_score(y_test, baseline_preds)
print(f"Baseline Accuracy: {baseline_acc * 100:.2f}%")

Baseline Accuracy: 95.61%


In [3]:
from sklearn.model_selection import GridSearchCV

# Define the search space for the 3 target hyperparameters
param_grid = {
    'n_estimators': [50, 100, 200],      # Number of boosting rounds
    'max_depth': [3, 5, 7],               # Tree depth (controls complexity)
    'learning_rate': [0.01, 0.05, 0.1]    # Step size shrinkage (eta)
}

grid_search = GridSearchCV(
    estimator=xgb.XGBClassifier(eval_metric="logloss", random_state=42),
    param_grid=param_grid,
    scoring='accuracy',
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print(f"Best CV Score: {grid_search.best_score_ * 100:.2f}%")

Fitting 5 folds for each of 27 candidates, totalling 135 fits
Best Parameters: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200}
Best CV Score: 97.58%


In [4]:
best_model = grid_search.best_estimator_
tuned_preds = best_model.predict(X_test)
tuned_acc = accuracy_score(y_test, tuned_preds)

print(f"Baseline Accuracy: {baseline_acc * 100:.2f}%")
print(f"Tuned Test Accuracy: {tuned_acc * 100:.2f}%")
print("\nClassification Report:\n", classification_report(y_test, tuned_preds))

# Save the trained model in Colab's workspace
best_model.save_model("xgb_model.json")
print("Model saved to xgb_model.json")

Baseline Accuracy: 95.61%
Tuned Test Accuracy: 94.74%

Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.90      0.93        42
           1       0.95      0.97      0.96        72

    accuracy                           0.95       114
   macro avg       0.95      0.94      0.94       114
weighted avg       0.95      0.95      0.95       114

Model saved to xgb_model.json
